# Tratamento Inicial dos Datasets

Este notebook aplica o mesmo tratamento inicial do projeto de regressão logística:
- Cria janelas futuras (3, 7, 15, 30 dias) para o preço de fechamento
- Gera rótulos binários (alta/baixa) para cada horizonte
- Remove linhas com NaN geradas pelos shifts
- Salva dois conjuntos tratados:
  - Regressão: valores originais + fechamentos futuros
  - Classificação: valores originais + targets futuros


In [ ]:
import os
from pathlib import Path
import pandas as pd

print('Bibliotecas carregadas com sucesso.')

## 1. Configuração de pastas

In [ ]:
BASE_DIR = Path('..').resolve()  # Base/
DATASETS_DIR = BASE_DIR / 'datasets'
OUTPUT_BASE_DIR = DATASETS_DIR / 'datasets_base'
OUTPUT_REGRESSAO_DIR = OUTPUT_BASE_DIR / 'regressao'
OUTPUT_CLASSIFICACAO_DIR = OUTPUT_BASE_DIR / 'classificacao'

OUTPUT_REGRESSAO_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_CLASSIFICACAO_DIR.mkdir(parents=True, exist_ok=True)

print('Pasta de entrada:', DATASETS_DIR)
print('Pasta de saída (regressão):', OUTPUT_REGRESSAO_DIR)
print('Pasta de saída (classificação):', OUTPUT_CLASSIFICACAO_DIR)

## 2. Função de tratamento

In [ ]:
def tratar_dataset(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    # Criação de janelas futuras para o preço de fechamento
    df = df.copy()
    df['Close_3d_fut'] = df['Close'].shift(-3)
    df['Close_7d_fut'] = df['Close'].shift(-7)
    df['Close_15d_fut'] = df['Close'].shift(-15)
    df['Close_30d_fut'] = df['Close'].shift(-30)

    # Remove linhas com NaN nas janelas futuras
    df = df.dropna(subset=['Close_3d_fut', 'Close_7d_fut', 'Close_15d_fut', 'Close_30d_fut'])

    # Converte coluna de data e define índice, se existir
    if 'Date' in df.columns:
        df['Date'] = pd.to_datetime(df['Date'])
        df = df.set_index('Date')

    # Cria rótulos binários (1 = alta, 0 = baixa/estável)
    df['target_3d'] = (df['Close_3d_fut'] > df['Close']).astype(int)
    df['target_7d'] = (df['Close_7d_fut'] > df['Close']).astype(int)
    df['target_15d'] = (df['Close_15d_fut'] > df['Close']).astype(int)
    df['target_30d'] = (df['Close_30d_fut'] > df['Close']).astype(int)

    future_cols = ['Close_3d_fut', 'Close_7d_fut', 'Close_15d_fut', 'Close_30d_fut']
    target_cols = ['target_3d', 'target_7d', 'target_15d', 'target_30d']

    # Regressão: mantém valores originais + fechamentos futuros
    df_regressao = df.drop(columns=target_cols)

    # Classificação: mantém valores originais + targets (remove fechamentos futuros)
    df_classificacao = df.drop(columns=future_cols)

    return df_regressao, df_classificacao

## 3. Processamento em lote

In [ ]:
csv_files = sorted(DATASETS_DIR.glob('*.csv'))
if not csv_files:
    raise FileNotFoundError(f'Nenhum CSV encontrado em {DATASETS_DIR}')

print(f'Arquivos encontrados: {len(csv_files)}')
for f in csv_files:
    print('-', f.name)

In [ ]:
for file_path in csv_files:
    print('' + '=' * 80)
    print(f'Processando: {file_path.name}')

    df = pd.read_csv(file_path)
    print('Shape original:', df.shape)

    # Validação mínima
    required_cols = {'Close'}
    missing_cols = required_cols - set(df.columns)
    if missing_cols:
        print(f'ERRO: colunas ausentes {missing_cols}. Arquivo ignorado.')
        continue

    df_regressao, df_classificacao = tratar_dataset(df)
    print('Shape tratado (regressão):', df_regressao.shape)
    print('Shape tratado (classificação):', df_classificacao.shape)

    # Salvar arquivos tratados
    output_name = file_path.stem + '_tratado.csv'
    output_path_reg = OUTPUT_REGRESSAO_DIR / output_name
    output_path_clf = OUTPUT_CLASSIFICACAO_DIR / output_name

    df_regressao.to_csv(output_path_reg, index=True)
    df_classificacao.to_csv(output_path_clf, index=True)

    print('Salvo (regressão) em:', output_path_reg)
    print('Salvo (classificação) em:', output_path_clf)

## 4. Resumo final

In [ ]:
processed_reg = sorted(OUTPUT_REGRESSAO_DIR.glob('*_tratado.csv'))
processed_clf = sorted(OUTPUT_CLASSIFICACAO_DIR.glob('*_tratado.csv'))

print('Arquivos tratados gerados (regressão):')
for f in processed_reg:
    print('-', f.name)

print('\nArquivos tratados gerados (classificação):')
for f in processed_clf:
    print('-', f.name)